In [1]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [2]:
SEED = 42
np.random.seed(SEED)


In [3]:
# =========================================================
# 1) TEMPERATURE REGRESSION
# =========================================================
exp_temp_encoding = Experiment(
    name="temperature_regression_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="temperature",
        target_mode="regression",
        # target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=("temperature",),
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=[
            # pojedyncze miasta
            ("Vancouver",),
            ("New York",),
            ("Chicago",),
            ("Miami",),
            ("Jerusalem",),

            # Południe USA
            ("Atlanta", "Nashville", "Charlotte", "Jacksonville", "Miami"),

            # Teksas
            ("Dallas", "Houston", "San Antonio"),

            # Izrael – wszystkie miasta
            ("Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem"),

            # USA – wszystkie miasta
            (
                "Vancouver", "Portland", "San Francisco", "Seattle",
                "Phoenix", "Albuquerque", "Denver", "San Antonio", "Dallas", "Houston",
                "Kansas City", "Minneapolis", "Saint Louis", "Chicago", "Nashville",
                "Indianapolis", "Atlanta", "Detroit", "Jacksonville", "Charlotte",
                "Miami", "Pittsburgh", "Philadelphia", "New York", "Boston"
            ),

            # wszystkie miasta razem
            (
                "Vancouver", "Portland", "San Francisco", "Seattle",
                "Phoenix", "Albuquerque", "Denver", "San Antonio", "Dallas", "Houston",
                "Kansas City", "Minneapolis", "Saint Louis", "Chicago", "Nashville",
                "Indianapolis", "Atlanta", "Detroit", "Jacksonville", "Charlotte",
                "Miami", "Pittsburgh", "Toronto", "Philadelphia", "New York",
                "Montreal", "Boston",
                "Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem"
            ),
        ]
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="regression",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(32, 64),
        loss="huber",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

# =========================================================
# 2) WIND (>=6 m/s) BINARY CLASSIFICATION
# =========================================================
exp_wind_encoding = Experiment(
    name="wind6_binary_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=("wind_speed",),
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=[
            # pojedyncze miasta
            ("Vancouver",),
            ("New York",),
            ("Chicago",),
            ("Miami",),
            ("Jerusalem",),

            # Południe USA
            ("Atlanta", "Nashville", "Charlotte", "Jacksonville", "Miami"),

            # Teksas
            ("Dallas", "Houston", "San Antonio"),

            # Izrael – wszystkie miasta
            ("Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem"),

            # USA – wszystkie miasta
            (
                "Vancouver", "Portland", "San Francisco", "Seattle",
                "Phoenix", "Albuquerque", "Denver", "San Antonio", "Dallas", "Houston",
                "Kansas City", "Minneapolis", "Saint Louis", "Chicago", "Nashville",
                "Indianapolis", "Atlanta", "Detroit", "Jacksonville", "Charlotte",
                "Miami", "Pittsburgh", "Philadelphia", "New York", "Boston"
            ),

            # wszystkie miasta razem
            (
                "Vancouver", "Portland", "San Francisco", "Seattle",
                "Phoenix", "Albuquerque", "Denver", "San Antonio", "Dallas", "Houston",
                "Kansas City", "Minneapolis", "Saint Louis", "Chicago", "Nashville",
                "Indianapolis", "Atlanta", "Detroit", "Jacksonville", "Charlotte",
                "Miami", "Pittsburgh", "Toronto", "Philadelphia", "New York",
                "Montreal", "Boston",
                "Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem"
            ),
        ]
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(128, 64),
        loss="binary_cross_entropy",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

experiments = [exp_temp_encoding, exp_wind_encoding]


In [4]:
search = Search()

results1 = search.run(exp_temp_encoding)



Starting experiment: temperature_regression_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 593.35it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 696.97it/s]



Configuration run 1/10:
WEATHER (variable):
  - cities: ('Vancouver',)

Training model


...

  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 691.45it/s]
Portland | windows: 100%|██████████| 361/361 [00:00<00:00, 639.43it/s]
San Francisco | windows: 100%|██████████| 361/361 [00:00<00:00, 624.15it/s]
Seattle | windows: 100%|██████████| 361/361 [00:00<00:00, 649.87it/s]
Phoenix | windows: 100%|██████████| 361/361 [00:00<00:00, 648.80it/s]
Albuquerque | windows: 100%|██████████| 361/361 [00:00<00:00, 640.84it/s]
Denver | windows: 100%|██████████| 361/361 [00:00<00:00, 642.69it/s]
San Antonio | windows: 100%|██████████| 361/361 [00:00<00:00, 552.93it/s]
Dallas | windows: 100%|██████████| 361/361 [00:00<00:00, 498.31it/s]
Houston | windows: 100%|██████████| 361/361 [00:00<00:00, 537.73it/s]
Kansas City | windows: 100%|██████████| 361/361 [00:00<00:00, 569.56it/s]
Minneapolis | windows: 100%|██████████| 361/361 [00:00<00:00, 636.36it/s]
Saint Louis | windows: 100%|██████████| 361/361 [00:00<00:00, 629.05it/s]
Chicago | windows: 100%|██████████| 361/361 [00:00<00:00, 623.6


Configuration run 10/10:
WEATHER (variable):
  - cities: ('Vancouver', 'Portland', 'San Francisco', 'Seattle', 'Phoenix', 'Albuquerque', 'Denver', 'San Antonio', 'Dallas', 'Houston', 'Kansas City', 'Minneapolis', 'Saint Louis', 'Chicago', 'Nashville', 'Indianapolis', 'Atlanta', 'Detroit', 'Jacksonville', 'Charlotte', 'Miami', 'Pittsburgh', 'Toronto', 'Philadelphia', 'New York', 'Montreal', 'Boston', 'Beersheba', 'Tel Aviv District', 'Eilat', 'Haifa', 'Nahariyya', 'Jerusalem')

Training model


Training:  15%|█▌        | 60/400 [00:43<04:04,  1.39it/s, acc=n/a, loss=3.2646, lr=0.00547157]

Early stopping at epoch 61, best val_loss=1.294172 after 50 epochs without improvement.
Training finished in 43.10 seconds

Experiment finished | total runs = 10



In [8]:
from IPython.core.display import HTML

for run in results1:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.65:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.60:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.58:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.55:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"AUC      : {metrics['auc']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {auc_color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>AUC</b>: {auc_val:.4f}
            </div>
            """
        ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        # display(HTML(
        #     f"""
        #     <div style="
        #         font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
        #         font-size: 13px;
        #         color: {color};
        #         padding-left: 12px;
        #         margin: 4px 0;
        #     ">
        #         <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
        #     </div>
        #     """
        # ))
        print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - cities: ('Vancouver',)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.9234
MSE              : 6.4342
RMSE             : 2.5366
Accuracy |err|≤2°C   : 0.6189


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - cities: ('New York',)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 3.4160
MSE              : 21.1247
RMSE             : 4.5962
Accuracy |err|≤2°C   : 0.4146


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - cities: ('Chicago',)
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 3.8067
MSE    

In [6]:
search = Search()

results2 = search.run(exp_wind_encoding)


Starting experiment: wind6_binary_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 643.83it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 630.85it/s]



Configuration run 1/10:
WEATHER (variable):
  - cities: ('Vancouver',)

Training model


...

  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 410.75it/s]
Portland | windows: 100%|██████████| 361/361 [00:00<00:00, 413.27it/s]
San Francisco | windows: 100%|██████████| 361/361 [00:00<00:00, 486.00it/s]
Seattle | windows: 100%|██████████| 361/361 [00:00<00:00, 511.25it/s]
Phoenix | windows: 100%|██████████| 361/361 [00:00<00:00, 562.31it/s]
Albuquerque | windows: 100%|██████████| 361/361 [00:00<00:00, 602.69it/s]
Denver | windows: 100%|██████████| 361/361 [00:00<00:00, 610.43it/s]
San Antonio | windows: 100%|██████████| 361/361 [00:00<00:00, 554.15it/s]
Dallas | windows: 100%|██████████| 361/361 [00:00<00:00, 583.46it/s]
Houston | windows: 100%|██████████| 361/361 [00:00<00:00, 584.01it/s]
Kansas City | windows: 100%|██████████| 361/361 [00:00<00:00, 604.16it/s]
Minneapolis | windows: 100%|██████████| 361/361 [00:00<00:00, 611.31it/s]
Saint Louis | windows: 100%|██████████| 361/361 [00:00<00:00, 549.02it/s]
Chicago | windows: 100%|██████████| 361/361 [00:00<00:00, 520.1


Configuration run 10/10:
WEATHER (variable):
  - cities: ('Vancouver', 'Portland', 'San Francisco', 'Seattle', 'Phoenix', 'Albuquerque', 'Denver', 'San Antonio', 'Dallas', 'Houston', 'Kansas City', 'Minneapolis', 'Saint Louis', 'Chicago', 'Nashville', 'Indianapolis', 'Atlanta', 'Detroit', 'Jacksonville', 'Charlotte', 'Miami', 'Pittsburgh', 'Toronto', 'Philadelphia', 'New York', 'Montreal', 'Boston', 'Beersheba', 'Tel Aviv District', 'Eilat', 'Haifa', 'Nahariyya', 'Jerusalem')

Training model


Training:  24%|██▍       | 96/400 [03:28<10:59,  2.17s/it, acc=0.6756, loss=0.5930, lr=0.00381047]


Early stopping at epoch 97, best val_loss=0.572020, train_acc=0.6756, val_acc=0.6794 after 50 epochs without improvement.
Training finished in 208.41 seconds

Experiment finished | total runs = 10



In [9]:
from IPython.core.display import HTML

for run in results2:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.60:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.55:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.53:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.51:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"Auc      : {metrics['auc']:.4f}")
        # display(HTML(
        #     f"""
        #     <div style="
        #         font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
        #         font-size: 13px;
        #         color: {auc_color};
        #         padding-left: 12px;
        #         margin: 4px 0;
        #     ">
        #         <b>AUC</b>: {auc_val:.4f}
        #     </div>
        #     """
        # ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
            </div>
            """
        ))
        print(f"Accuracy |err|≤2°C : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - cities: ('Vancouver',)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5427
Precision: 0.6061
Recall   : 0.6250
Auc      : 0.5144


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - cities: ('New York',)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.6494
Precision: 0.6561
Recall   : 0.7880
Auc      : 0.6915


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - cities: ('Chicago',)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.7119
Precision: 0.7658
Recall   : 0.8897
Auc      : 0.5393


EXPERIMENT: wind6_binary_encoding
TASK: B